# 00 · Simulate a slice and find spatial clusters

In this tutorial, you will use FEAST to generate new expression counts for DLPFC slice **151675**.
You will then use GraphST to group spots into spatial clusters and compare those clusters with the known tissue layers.

Start with [data and environment setup](README.md#setup). You need `dlpfc/151675.h5ad`, `scikit-misc`, and a GraphST environment with CUDA, R and mclust.
Notebooks 02 and 04 use this same slice. Open the notebook from `FEAST/` or `FEAST/tutorial/`.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import FEAST

TUTORIAL = Path.cwd() if Path.cwd().name == "tutorial" else Path.cwd() / "tutorial"
sys.path.insert(0, str(TUTORIAL))
from _utils import data_root, load_counts, gene_summary, gene_values, spatial_panel
DATA = data_root()

OUT = TUTORIAL / "outputs" / "00"
OUT.mkdir(parents=True, exist_ok=True)
print("FEAST", FEAST.__version__)

### 1. Load the reference slice

Start with raw counts: the number of transcripts measured for each gene at each spot.
FEAST uses these counts to learn how much each gene is expressed, how much its expression varies, and how often its count is zero.
Normalizing the counts first would change what FEAST learns.

The tissue-layer labels in `obs['ground_truth']` will help us evaluate the clusters later.

In [ ]:
reference = load_counts(DATA / "dlpfc/151675.h5ad", "ground_truth")
print(f"{reference.n_obs:,} spots × {reference.n_vars:,} genes")

### 2. Generate new expression counts

FEAST keeps the reference spot locations and generates a new count matrix.
The two settings below control how it does this:

- `parameter_mode="hungarian"` learns a distribution of gene statistics, samples new statistics, and matches them to genes.
- `spatial_mode="reference_rank"` uses each gene's ordering of high- and low-expression spots to arrange the new counts across the tissue.

We use seed **2026** and the global SciPy assignment, following the single-slice reproduction workflow.
This full-slice example can require several GB of memory.

In [ ]:
%%capture --no-stderr
simulated = FEAST.simulate(
    reference, seed=2026, parameter_mode="hungarian", spatial_mode="reference_rank",
    assignment_solver="scipy", assignment_blocks=False, verbose=False,
)

In [ ]:
assert simulated.obs_names.equals(reference.obs_names)
assert np.array_equal(simulated.obsm["spatial"], reference.obsm["spatial"])
ref_stats, sim_stats = gene_summary(reference), gene_summary(simulated)
fig, axes = plt.subplots(1, 3, figsize=(10, 3), layout="constrained")
for ax, key in zip(axes, ref_stats.columns):
    x, y = ref_stats[key], sim_stats[key]
    if key != "zero_fraction":
        x, y = np.log1p(x), np.log1p(y)
    ax.scatter(x, y, s=2, alpha=0.3)
    limits = [min(x.min(), y.min()), max(x.max(), y.max())]
    ax.plot(limits, limits, color="black", linewidth=0.7)
    ax.set(title=key, xlabel="Reference", ylabel="Simulated")
plt.show()

### 3. Prepare the expression data for GraphST

Choose 3,000 variable genes from the **original slice**, then use those same genes in the simulated slice.
This keeps gene selection independent of the simulated result.

For GraphST, normalize each spot to a total count of 10,000 and apply `log1p` once.
These steps operate on a separate copy, so the original and simulated counts remain available.

In [ ]:
import scanpy as sc
panel_source = reference.copy()
sc.pp.highly_variable_genes(panel_source, n_top_genes=3000, flavor="seurat_v3")
genes = panel_source.var_names[panel_source.var["highly_variable"]]
prepared = ad.AnnData(
    X=simulated[:, genes].X.copy(),
    obs=pd.DataFrame({"ground_truth": pd.Categorical(simulated.obs["ground_truth"])},
                     index=simulated.obs_names),
    var=pd.DataFrame(index=genes),
)
prepared.obsm["spatial"] = simulated.obsm["spatial"].copy()
sc.pp.normalize_total(prepared, target_sum=1e4)
sc.pp.log1p(prepared)
prepared.write_h5ad(OUT / "graphst_input.h5ad")

### 4. Run GraphST

GraphST learns a representation of each spot from its expression and spatial neighbors.
We then use mclust to group those representations into clusters.
The main calls in [graphst_step.py](graphst_step.py) are:

```python
model = graphst_module.GraphST(data, device=device, epochs=600, random_seed=2026)
result = model.train()
clustering(result, n_clusters=n_clusters, method="mclust", radius=50, refinement=True)
```

The script runs in the separate GraphST environment. It scales the prepared expression data and uses the gene panel selected above.
We supply the number of known tissue layers as the requested cluster count, following Study 01.
The layer labels themselves are not used to train GraphST.

GraphST uses seed 2026 for training; its clustering utility uses seeds 42 for PCA and 2020 for mclust.

In [ ]:
import os
import subprocess
with (OUT / "graphst.log").open("w") as log:
    subprocess.run([
        os.environ["GRAPHST_PYTHON"], str(TUTORIAL / "graphst_step.py"),
        str(OUT / "graphst_input.h5ad"), str(OUT / "graphst_result.h5ad"), "--device", "cuda",
    ], check=True, stdout=log, stderr=subprocess.STDOUT)
clustered = ad.read_h5ad(OUT / "graphst_result.h5ad")

In [ ]:
from sklearn.metrics import adjusted_rand_score
ari = adjusted_rand_score(clustered.obs["ground_truth"], clustered.obs["predicted_cluster"])
fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
spatial_panel(axes[0], clustered.obsm["spatial"], clustered.obs["ground_truth"], "Known layers", True)
spatial_panel(axes[1], clustered.obsm["spatial"], clustered.obs["predicted_cluster"], f"GraphST · ARI {ari:.3f}", True)
plt.show()

**How to read the plots:** in the expression checks, points near the diagonal indicate similar gene statistics in the original and simulated slices.
In the spatial maps, compare the shapes of the tissue regions rather than their colors: cluster names and colors do not correspond directly to layer names.

The adjusted Rand index (ARI) measures agreement between the clusters and known layers. A value of 1 means perfect agreement; values near 0 indicate agreement close to chance.
This score describes this one simulation, with the cluster count supplied in advance.

### Run status

The 151675 example has not yet been run end to end. Run the cells above to generate its plots and clustering result.